# 🔮 AI-Based Software Defect Prediction with Bayesian Model Averaging
**Course syllabus topic:** Bayesian Model Averaging · Boosting · Adaptive Basis Function Models · CART · GAMs · Ensemble Learning
**Problem:** Identify defect-prone software modules **while quantifying model uncertainty**.
**Data:** NASA / PROMISE MDP software-defect datasets (module-level static code metrics).
**Primary algorithm:** Bayesian Model Averaging (BMA) · **Comparison:** Logistic Regression, Random Forest, Naive Bayes
**Evaluation:** ROC-AUC, F1, Brier score, log-loss (+ PR-AUC, ECE calibration)

> ▶ **Run all cells (Runtime → Run all).** The last cell launches the glassmorphism **Streamlit web demo** and prints a public link.

| Section | Content |
|---|---|
| 1 | Setup & data |
| 2 | Exploration & preprocessing (fit on train only) |
| 3 | Bayesian Model Averaging — candidate models, posterior weights, averaging |
| 4 | Uncertainty plots & defect-risk output |
| 5 | Calibration |
| 6 | Model-comparison table (same held-out test set) |
| 7 | Robustness: repeated splits + all 9 NASA datasets |
| 8 | Conclusions |
| 9 | 🚀 Web demo (Streamlit, glassmorphism UI) |

## 1 · Setup & data

In [ ]:
!pip -q install streamlit plotly 2>&1 | tail -1

In [ ]:
%%writefile bma.py
"""
bma.py  -  Bayesian Model Averaging for software defect prediction
===================================================================
Core library used by the notebook and by the Streamlit app.

Pipeline
--------
1. load_dataset()      NASA MDP / PROMISE module-level metrics (local CSV -> GitHub fallback)
2. Preprocessor        signed-log1p -> drop constant -> drop correlated -> standardise
                       (fitted ONLY on training data)
3. BMAClassifier       candidate probabilistic models + BIC posterior model weights +
                       Laplace parameter posteriors -> averaged predictive distribution
4. run_study()         same held-out test set for every algorithm, tuning on validation only
5. repeated_study()    robustness over several random splits
"""
from __future__ import annotations

import io
import os
import time
import urllib.request
import warnings

import numpy as np
import pandas as pd
from scipy.special import expit
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score,
                             log_loss, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

TARGET = "defective"
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()

DATASETS = {
    "KC1": "NASA KC1 - C++ storage-management system",
    "PC1": "NASA PC1 - flight software for an earth-orbiting satellite",
    "PC3": "NASA PC3 - flight software for an earth-orbiting satellite",
    "PC4": "NASA PC4 - flight software for an earth-orbiting satellite",
    "CM1": "NASA CM1 - spacecraft instrument in C",
    "JM1": "NASA JM1 - real-time predictive ground system",
    "KC3": "NASA KC3 - storage management for ground data",
    "MW1": "NASA MW1 - zero-gravity experiment (combustion)",
    "MC2": "NASA MC2 - video guidance system",
}
_RAW = ("https://raw.githubusercontent.com/klainfo/NASADefectDataset/master/"
        "CleanedData/MDP/D%27/{n}.arff")


# ----------------------------------------------------------------------------
# 1. DATA
# ----------------------------------------------------------------------------
def _arff_to_df(text: str) -> pd.DataFrame:
    from scipy.io import arff
    data, _ = arff.loadarff(io.StringIO(text))
    df = pd.DataFrame(data)
    t = df.columns[-1]
    s = df[t].map(lambda v: v.decode() if isinstance(v, bytes) else str(v)).str.strip()
    df = df.drop(columns=[t])
    df[TARGET] = s.isin(["Y", "yes", "true", "TRUE", "1"]).astype(int)
    return df


def load_dataset(name: str = "KC1", data_dir: str | None = None) -> pd.DataFrame:
    """Load a NASA MDP dataset: local CSV first, then the public GitHub mirror."""
    name = name.upper()
    for d in filter(None, [data_dir, os.path.join(HERE, "data"), os.path.join(os.getcwd(), "data")]):
        p = os.path.join(d, f"{name}.csv")
        if os.path.exists(p):
            return pd.read_csv(p)
    with urllib.request.urlopen(_RAW.format(n=name), timeout=60) as r:
        df = _arff_to_df(r.read().decode("utf-8", "ignore"))
    os.makedirs(os.path.join(HERE, "data"), exist_ok=True)
    try:
        df.to_csv(os.path.join(HERE, "data", f"{name}.csv"), index=False)
    except OSError:
        pass
    return df


# ----------------------------------------------------------------------------
# 2. PREPROCESSING  (fit on training data only)
# ----------------------------------------------------------------------------
class Preprocessor:
    """signed log1p -> drop constant cols -> drop |corr|>thr cols -> z-score."""

    def __init__(self, corr_threshold: float = 0.95):
        self.corr_threshold = corr_threshold

    @staticmethod
    def _slog(a):
        return np.sign(a) * np.log1p(np.abs(a))

    def fit(self, X: pd.DataFrame):
        self.raw_columns_ = list(X.columns)
        Z = pd.DataFrame(self._slog(X.values.astype(float)), columns=X.columns)
        keep = [c for c in Z.columns if Z[c].std() > 1e-9]
        corr = Z[keep].corr().abs().fillna(0)
        dropped, final = set(), []
        for c in keep:                       # greedy: keep the first of a correlated pair
            if c in dropped:
                continue
            final.append(c)
            for o in keep:
                if o != c and o not in dropped and o not in final and corr.loc[c, o] > self.corr_threshold:
                    dropped.add(o)
        self.columns_ = final
        self.dropped_constant_ = [c for c in X.columns if c not in keep]
        self.dropped_correlated_ = sorted(dropped)
        self.mean_ = Z[final].mean().values
        self.std_ = Z[final].std().values
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        Z = self._slog(X[self.columns_].values.astype(float))
        return (Z - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)


# ----------------------------------------------------------------------------
# 3. BAYESIAN MODEL AVERAGING
# ----------------------------------------------------------------------------
def fit_logit(X, y, l2=1.0, max_iter=60):
    """Bayesian logistic regression, MAP via IRLS with N(0, 1/l2) prior.
    Returns (beta, loglik, posterior_covariance[Laplace])."""
    n, d = X.shape
    Xd = np.hstack([np.ones((n, 1)), X])
    pen = np.full(d + 1, float(l2))
    pen[0] = 1e-6
    beta = np.zeros(d + 1)
    for _ in range(max_iter):
        p = expit(np.clip(Xd @ beta, -30, 30))
        w = p * (1 - p) + 1e-9
        g = Xd.T @ (y - p) - pen * beta
        H = (Xd * w[:, None]).T @ Xd + np.diag(pen)
        step = np.linalg.solve(H, g)
        step = np.clip(step, -5, 5)
        beta = beta + step
        if np.max(np.abs(step)) < 1e-7:
            break
    p = expit(np.clip(Xd @ beta, -30, 30))
    w = p * (1 - p) + 1e-9
    H = (Xd * w[:, None]).T @ Xd + np.diag(pen)
    ll = float(np.sum(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)))
    return beta, ll, np.linalg.inv(H)


def _entropy(p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


class BMAClassifier:
    """Bayesian Model Averaging over probabilistic defect classifiers.

    Candidate models (M_k)
      * Bayesian logistic regressions over many subsets of the code metrics, found by
        greedy BIC forward selection + MC3 (Markov-chain model composition) search
        -> variable-selection uncertainty
      * Gaussian Naive Bayes (all metrics)
      * (optional) shallow decision trees with Laplace-smoothed leaves

    Posterior model weights  P(M_k | D)  ∝  P(M_k) exp(-tau * BIC_k / 2)
    (Schwarz / BIC approximation to the marginal likelihood, Raftery 1995).

    Predictive distribution
      p(y=1|x,D) = sum_k P(M_k|D) * E[p(y=1|x,theta_k,M_k) | D]
    Logistic models carry a Laplace (Gaussian) posterior over coefficients, so every
    prediction also gets a credible interval; uncertainty is decomposed into
      * aleatoric  (expected entropy of the members)
      * epistemic  (mutual information = total entropy - aleatoric)
      * between-model vs within-model variance.
    """

    def __init__(self, max_features=10, n_mc3=500, include_nb=True, include_trees=False,
                 tree_depths=(2, 3, 4), temperature=1.0, prior_precision=1.0, max_models=60,
                 n_draws=2000, random_state=0):
        self.max_features = max_features
        self.n_mc3 = n_mc3
        self.include_nb = include_nb
        self.include_trees = include_trees
        self.tree_depths = tree_depths
        self.temperature = temperature
        self.prior_precision = prior_precision
        self.max_models = max_models
        self.n_draws = n_draws
        self.random_state = random_state

    # ---- fitting -----------------------------------------------------------
    def fit(self, X, y, feature_names=None):
        X = np.asarray(X, float)
        y = np.asarray(y, float)
        n, p = X.shape
        rng = np.random.default_rng(self.random_state)
        self.feature_names_ = list(feature_names) if feature_names is not None else [f"x{i}" for i in range(p)]
        self.n_train_ = n
        logn = np.log(n)
        cache = {}                                   # subset (sorted tuple) -> fitted logistic model

        def get(S):
            key = tuple(sorted(S))
            if key not in cache:
                beta, ll, cov = fit_logit(X[:, list(key)], y, self.prior_precision, max_iter=25)
                cache[key] = dict(kind="logreg", idx=list(key), beta=beta, cov=cov, loglik=ll,
                                  k=len(key) + 1, bic=-2 * ll + (len(key) + 1) * logn)
            return cache[key]

        # (a1) greedy BIC forward selection -> good starting point
        chosen, remaining = [], list(range(p))
        get([])
        for _ in range(min(self.max_features, p)):
            j = min(remaining, key=lambda j: get(chosen + [j])["bic"])
            chosen.append(j)
            remaining.remove(j)
        # (a2) MC3: Metropolis search over metric subsets  (add / drop / swap moves)
        cur = set(get(chosen)["idx"])
        cur_bic = get(cur)["bic"]
        for _ in range(self.n_mc3):
            moves = []
            if len(cur) < self.max_features:
                moves.append("add")
            if len(cur) > 0:
                moves.append("drop")
            if 0 < len(cur) < p:
                moves.append("swap")
            mv = rng.choice(moves)
            prop = set(cur)
            if mv == "add":
                prop.add(int(rng.choice([j for j in range(p) if j not in cur])))
            elif mv == "drop":
                prop.remove(int(rng.choice(sorted(cur))))
            else:
                prop.remove(int(rng.choice(sorted(cur))))
                prop.add(int(rng.choice([j for j in range(p) if j not in cur])))
            pb = get(prop)["bic"]
            if np.log(rng.random() + 1e-300) < -0.5 * (pb - cur_bic):
                cur, cur_bic = prop, pb
        models = sorted(cache.values(), key=lambda m: m["bic"])[: self.max_models]
        # (b) Gaussian naive Bayes
        if self.include_nb:
            nb = GaussianNB(var_smoothing=1e-3).fit(X, y)
            pr = nb.predict_proba(X)
            ll = float(np.sum(np.log(np.where(y == 1, pr[:, 1], pr[:, 0]) + 1e-12)))
            models.append(dict(kind="nb", est=nb, loglik=ll, k=2 * p + 1, idx=list(range(p)),
                               bic=-2 * ll + (2 * p + 1) * logn))
        # (c) optional shallow trees with Laplace-smoothed leaves
        if self.include_trees:
            for d in self.tree_depths:
                t = DecisionTreeClassifier(max_depth=d, min_samples_leaf=10, random_state=self.random_state).fit(X, y)
                leaves = t.apply(X)
                tab = {lf: (y[leaves == lf].sum() + 1.0) / ((leaves == lf).sum() + 2.0) for lf in np.unique(leaves)}
                pr = np.array([tab[l] for l in leaves])
                ll = float(np.sum(y * np.log(pr) + (1 - y) * np.log(1 - pr)))
                models.append(dict(kind="tree", est=t, tab=tab, loglik=ll, k=int(t.get_n_leaves()),
                                   idx=None, bic=-2 * ll + int(t.get_n_leaves()) * logn))
        models.sort(key=lambda m: m["bic"])
        for r, m in enumerate(models, 1):
            m["name"] = f"M{r:02d}-" + {"logreg": "LR", "nb": "NB", "tree": "Tree"}[m["kind"]]
        self.models_ = models
        self.n_visited_ = len(cache)
        self._compute_weights()
        return self

    def _compute_weights(self):
        bic = np.array([m["bic"] for m in self.models_])
        a = -0.5 * self.temperature * (bic - bic.min())
        w = np.exp(a - a.max())
        self.weights_ = w / w.sum()
        return self

    def set_temperature(self, tau: float):
        """tau=1 -> textbook BIC posterior; tau<1 -> flatter (more model uncertainty)."""
        self.temperature = float(tau)
        return self._compute_weights()

    # ---- inspection --------------------------------------------------------
    def weights_table(self) -> pd.DataFrame:
        rows = []
        for m, w in zip(self.models_, self.weights_):
            feats = ""
            if m["kind"] == "logreg":
                feats = ", ".join(self.feature_names_[i] for i in m["idx"])
            rows.append(dict(model=m["name"], kind=m["kind"], params=m["k"],
                             log_likelihood=m["loglik"], BIC=m["bic"], posterior_weight=w,
                             metrics=feats))
        return pd.DataFrame(rows).sort_values("posterior_weight", ascending=False).reset_index(drop=True)

    def inclusion_probabilities(self) -> pd.Series:
        """Posterior inclusion probability of each metric over the logistic candidates."""
        pip = np.zeros(len(self.feature_names_))
        tot = 0.0
        for m, w in zip(self.models_, self.weights_):
            if m["kind"] == "logreg":
                pip[m["idx"]] += w
                tot += w
        pip = pip / tot if tot > 0 else pip
        return pd.Series(pip, index=self.feature_names_).sort_values(ascending=False)

    @property
    def best_model_(self):
        return self.models_[int(np.argmax(self.weights_))]

    # ---- prediction --------------------------------------------------------
    def _member_proba(self, m, X):
        """Posterior-mean-ish prediction of one member (point estimate)."""
        if m["kind"] == "logreg":
            Xd = np.hstack([np.ones((len(X), 1)), X[:, m["idx"]]])
            return expit(Xd @ m["beta"])
        if m["kind"] == "nb":
            return m["est"].predict_proba(X)[:, 1]
        return np.array([m["tab"].get(l, 0.5) for l in m["est"].apply(X)])

    def predict_proba(self, X) -> np.ndarray:
        """BMA predictive probability (fast, no sampling)."""
        X = np.asarray(X, float)
        out = np.zeros(len(X))
        for m, w in zip(self.models_, self.weights_):
            if w > 1e-8:
                out += w * self._member_proba(m, X)
        return out

    def predict_proba_best(self, X) -> np.ndarray:
        return self._member_proba(self.best_model_, np.asarray(X, float))

    def predict_dist(self, X, level=0.90, n_draws=None, seed=None) -> dict:
        """Full predictive distribution with credible intervals and uncertainty split."""
        X = np.asarray(X, float)
        S = int(n_draws or self.n_draws)
        rng = np.random.default_rng(self.random_state if seed is None else seed)
        counts = rng.multinomial(S, self.weights_)
        draws, member_of = [], []
        for mi, (m, c) in enumerate(zip(self.models_, counts)):
            if c == 0:
                continue
            if m["kind"] == "logreg":
                betas = rng.multivariate_normal(m["beta"], m["cov"], size=c, method="cholesky")
                Xd = np.hstack([np.ones((len(X), 1)), X[:, m["idx"]]])
                dr = expit(np.clip(Xd @ betas.T, -30, 30))               # (N, c)
            else:
                dr = np.repeat(self._member_proba(m, X)[:, None], c, axis=1)
            draws.append(dr)
            member_of += [mi] * c
        D = np.hstack(draws)                                           # (N, S)
        member_of = np.array(member_of)
        mean = D.mean(1)
        a = (1 - level) / 2
        lo, hi = np.quantile(D, a, axis=1), np.quantile(D, 1 - a, axis=1)
        total_var = D.var(1)
        mm = np.stack([D[:, member_of == k].mean(1) for k in np.unique(member_of)], 1)
        wk = np.array([(member_of == k).mean() for k in np.unique(member_of)])
        between = (wk * (mm - mean[:, None]) ** 2).sum(1)
        within = np.maximum(total_var - between, 0)
        total_ent = _entropy(mean)
        aleatoric = _entropy(D).mean(1)
        epistemic = np.maximum(total_ent - aleatoric, 0)
        return dict(mean=mean, lo=lo, hi=hi, width=hi - lo, std=np.sqrt(total_var),
                    between_var=between, within_var=within, total_entropy=total_ent,
                    aleatoric=aleatoric, epistemic=epistemic, draws=D, level=level)


# ----------------------------------------------------------------------------
# 4. METRICS
# ----------------------------------------------------------------------------
def ece_score(y, p, n_bins=10):
    """Expected calibration error (equal-width bins)."""
    y, p = np.asarray(y), np.asarray(p)
    bins = np.minimum((p * n_bins).astype(int), n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = bins == b
        if m.any():
            e += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(e)


def reliability_curve(y, p, n_bins=8):
    """Equal-frequency reliability points: (mean predicted, observed freq, count)."""
    y, p = np.asarray(y), np.asarray(p)
    order = np.argsort(p)
    chunks = np.array_split(order, n_bins)
    return (np.array([p[c].mean() for c in chunks]), np.array([y[c].mean() for c in chunks]),
            np.array([len(c) for c in chunks]))


def best_f1_threshold(y, p):
    ts = np.unique(np.round(np.quantile(p, np.linspace(0.02, 0.98, 97)), 6))
    f = [f1_score(y, p >= t, zero_division=0) for t in ts]
    return float(ts[int(np.argmax(f))])


def metrics_at(y, p, thr):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return dict(ROC_AUC=roc_auc_score(y, p), PR_AUC=average_precision_score(y, p),
                F1=f1_score(y, p >= thr, zero_division=0),
                Precision=precision_score(y, p >= thr, zero_division=0),
                Recall=recall_score(y, p >= thr, zero_division=0),
                Brier=brier_score_loss(y, p), LogLoss=log_loss(y, p), ECE=ece_score(y, p))


INTERPRETABILITY = {
    "Bayesian Model Averaging": "High - posterior weights, inclusion probabilities, credible intervals",
    "Best single BIC model": "High - one sparse logistic model",
    "Logistic Regression": "High - coefficients / odds ratios",
    "Naive Bayes": "Medium - per-feature likelihoods, strong independence assumption",
    "Random Forest": "Low - ensemble of trees (feature importances only)",
}


# ----------------------------------------------------------------------------
# 5. FULL STUDY (identical held-out test set for every algorithm)
# ----------------------------------------------------------------------------
def make_splits(df, seed=42, test_size=0.2, val_size=0.2, target=TARGET):
    df = df.drop_duplicates().reset_index(drop=True)        # identical modules would leak across splits
    X, y = df.drop(columns=[target]), df[target].values
    Xtv, Xte, ytv, yte = train_test_split(X, y, test_size=test_size, stratify=y, random_state=seed)
    Xtr, Xva, ytr, yva = train_test_split(Xtv, ytv, test_size=val_size / (1 - test_size),
                                          stratify=ytv, random_state=seed)
    return dict(Xtr=Xtr, Xva=Xva, Xte=Xte, ytr=ytr, yva=yva, yte=yte, n_dedup=len(df))


def _timed(fn):
    t = time.perf_counter()
    out = fn()
    return out, time.perf_counter() - t


def run_study(df, seed=42, test_size=0.2, val_size=0.2, temperature=1.0, max_features=10,
              n_draws=2000, fast=False, target=TARGET) -> dict:
    sp = make_splits(df, seed, test_size, val_size, target)
    pre = Preprocessor().fit(sp["Xtr"])
    Ztr, Zva, Zte = (pre.transform(sp[k]) for k in ("Xtr", "Xva", "Xte"))
    ytr, yva, yte = sp["ytr"], sp["yva"], sp["yte"]
    names = pre.columns_
    probs, fit_t, inf_t, extra = {}, {}, {}, {}

    # --- Bayesian Model Averaging (no validation data needed to fit) ---
    bma, t = _timed(lambda: BMAClassifier(max_features=max_features, temperature=temperature,
                                          n_draws=n_draws, random_state=seed).fit(Ztr, ytr, names))
    fit_t["Bayesian Model Averaging"] = t
    pv = bma.predict_proba(Zva)
    pt, ti = _timed(lambda: bma.predict_proba(Zte))
    probs["Bayesian Model Averaging"] = (pv, pt)
    inf_t["Bayesian Model Averaging"] = ti
    probs["Best single BIC model"] = (bma.predict_proba_best(Zva), bma.predict_proba_best(Zte))
    fit_t["Best single BIC model"] = t
    inf_t["Best single BIC model"] = 0.0

    # --- Logistic Regression (C tuned on validation log-loss) ---
    def fit_lr():
        best = None
        for C in (0.01, 0.1, 1, 10, 100):
            m = LogisticRegression(C=C, max_iter=3000).fit(Ztr, ytr)
            s = log_loss(yva, m.predict_proba(Zva)[:, 1])
            if best is None or s < best[0]:
                best = (s, m, C)
        return best
    (_, lr, lrC), t = _timed(fit_lr)
    fit_t["Logistic Regression"] = t
    pt, ti = _timed(lambda: lr.predict_proba(Zte)[:, 1])
    probs["Logistic Regression"] = (lr.predict_proba(Zva)[:, 1], pt)
    inf_t["Logistic Regression"] = ti
    extra["lr"], extra["lr_C"] = lr, lrC

    # --- Naive Bayes (var_smoothing tuned on validation log-loss) ---
    def fit_nb():
        best = None
        for vs in (1e-9, 1e-6, 1e-3, 1e-1):
            m = GaussianNB(var_smoothing=vs).fit(Ztr, ytr)
            s = log_loss(yva, np.clip(m.predict_proba(Zva)[:, 1], 1e-6, 1 - 1e-6))
            if best is None or s < best[0]:
                best = (s, m)
        return best
    (_, nb), t = _timed(fit_nb)
    fit_t["Naive Bayes"] = t
    pt, ti = _timed(lambda: nb.predict_proba(Zte)[:, 1])
    probs["Naive Bayes"] = (nb.predict_proba(Zva)[:, 1], pt)
    inf_t["Naive Bayes"] = ti

    # --- Random Forest (small grid tuned on validation log-loss) ---
    def fit_rf():
        grid = [(6, 5)] if fast else [(4, 5), (8, 3), (None, 2), (None, 5)]
        best = None
        for depth, leaf in grid:
            m = RandomForestClassifier(n_estimators=150 if fast else 250, max_depth=depth,
                                       min_samples_leaf=leaf, n_jobs=-1, random_state=seed).fit(Ztr, ytr)
            s = log_loss(yva, np.clip(m.predict_proba(Zva)[:, 1], 1e-3, 1 - 1e-3))
            if best is None or s < best[0]:
                best = (s, m)
        return best
    (_, rf), t = _timed(fit_rf)
    fit_t["Random Forest"] = t
    pt, ti = _timed(lambda: rf.predict_proba(Zte)[:, 1])
    probs["Random Forest"] = (rf.predict_proba(Zva)[:, 1], pt)
    inf_t["Random Forest"] = ti
    extra["rf"] = rf

    # --- comparison table ---
    rows, thr = [], {}
    for name, (pv, pt) in probs.items():
        thr[name] = best_f1_threshold(yva, pv)
        mv, mt = metrics_at(yva, pv, thr[name]), metrics_at(yte, pt, thr[name])
        rows.append({"Algorithm": name, "Validation ROC-AUC": mv["ROC_AUC"], "Test ROC-AUC": mt["ROC_AUC"],
                     "Test PR-AUC": mt["PR_AUC"], "Test F1": mt["F1"], "Test Precision": mt["Precision"],
                     "Test Recall": mt["Recall"], "Test Brier": mt["Brier"], "Test Log-loss": mt["LogLoss"],
                     "Test ECE": mt["ECE"], "Training time (s)": fit_t[name],
                     "Inference time (ms/1000)": 1000 * inf_t[name] / len(yte) * 1000,
                     "Interpretability": INTERPRETABILITY[name]})
    table = pd.DataFrame(rows)
    return dict(table=table, probs=probs, thr=thr, bma=bma, pre=pre, splits=sp, extra=extra,
                Ztr=Ztr, Zva=Zva, Zte=Zte, ytr=ytr, yva=yva, yte=yte, seed=seed)


def repeated_study(df, seeds=range(5), **kw) -> pd.DataFrame:
    """Re-run the whole study on several random splits (robustness)."""
    out = []
    for s in seeds:
        r = run_study(df, seed=int(s), fast=True, **kw)
        t = r["table"].copy()
        t["seed"] = s
        out.append(t)
    return pd.concat(out, ignore_index=True)


def summarise_repeated(rep: pd.DataFrame, cols=("Test ROC-AUC", "Test F1", "Test Brier", "Test Log-loss", "Test ECE")):
    g = rep.groupby("Algorithm")[list(cols)]
    m, s = g.mean(), g.std()
    out = pd.DataFrame(index=m.index)
    for c in cols:
        out[c] = [f"{a:.3f} ± {b:.3f}" for a, b in zip(m[c], s[c])]
    return out.reset_index()


def score_modules(result: dict, raw_df: pd.DataFrame, level=0.90, n_draws=1000) -> pd.DataFrame:
    """Score arbitrary raw modules with BMA -> probability, credible interval, uncertainty, tier."""
    Z = result["pre"].transform(raw_df)
    d = result["bma"].predict_dist(Z, level=level, n_draws=n_draws)
    thr = result["thr"]["Bayesian Model Averaging"]
    out = pd.DataFrame({"p_defect": d["mean"], "ci_low": d["lo"], "ci_high": d["hi"],
                        "ci_width": d["width"], "epistemic_uncertainty": d["epistemic"],
                        "aleatoric_uncertainty": d["aleatoric"]}, index=raw_df.index)
    out["risk_tier"] = np.where(out.ci_low >= thr, "High", np.where(out.ci_high < thr, "Low", "Review"))
    return out


if __name__ == "__main__":
    d = load_dataset("KC1")
    r = run_study(d)
    pd.set_option("display.width", 250, "display.max_columns", 30)
    print(r["table"].drop(columns="Interpretability").round(4))
    print(r["bma"].weights_table().head(8))

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
from sklearn.metrics import roc_curve, roc_auc_score, brier_score_loss
import bma as B
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25,
                     "axes.prop_cycle": plt.cycler(color=["#7c3aed", "#06b6d4", "#ec4899", "#f59e0b", "#10b981", "#a78bfa"])})
pd.set_option("display.width", 220, "display.max_columns", 40)

DATASET = "PC4"      # try: KC1, PC1, PC3, CM1, JM1, KC3, MW1, MC2
SEED = 42
df = B.load_dataset(DATASET)      # downloads the public NASA MDP mirror on GitHub if not cached
print(f"{DATASET}: {df.shape[0]} modules, {df.shape[1]-1} code metrics, defect rate {df.defective.mean():.1%}")
df.head()

## 2 · Exploration & preprocessing
Exact duplicate modules are removed **before** splitting (they would leak between train and test). The split is stratified 60 / 20 / 20 (train / validation / test); the **same held-out test set is used for every algorithm**. Preprocessing (signed-log1p → drop constants → drop |ρ|>0.95 → z-score) is **fitted on training data only**.

In [ ]:
sp = B.make_splits(df, seed=SEED)
print("unique modules:", sp["n_dedup"], "| train/val/test:", len(sp["ytr"]), len(sp["yva"]), len(sp["yte"]))
print("defect rate  train/val/test:", sp["ytr"].mean().round(3), sp["yva"].mean().round(3), sp["yte"].mean().round(3))
pre = B.Preprocessor().fit(sp["Xtr"])
print(f"kept {len(pre.columns_)} metrics | dropped constant: {pre.dropped_constant_} | dropped correlated: {len(pre.dropped_correlated_)}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
Xall = sp["Xtr"]
cm = Xall[pre.columns_].corr()
im = ax[0].imshow(cm, cmap="viridis"); ax[0].set_title("Metric correlation (train, after pruning)"); ax[0].grid(False); plt.colorbar(im, ax=ax[0])
top = Xall[pre.columns_].corrwith(pd.Series(sp["ytr"], index=Xall.index), method="spearman").abs().sort_values().tail(10)
top.plot.barh(ax=ax[1], color="#7c3aed"); ax[1].set_title("|Spearman ρ| with defect label (top 10)")
plt.tight_layout(); plt.show()

## 3 · Bayesian Model Averaging
For a defect label *y* and module metrics *x*, BMA does **not** pick one model. It averages over a set of candidates {M₁ … M_K}:

$$p(y=1\mid x,D)=\sum_{k} P(M_k\mid D)\; p(y=1\mid x,M_k,D),\qquad P(M_k\mid D)\propto P(M_k)\,e^{-\tau\,\mathrm{BIC}_k/2}$$

* **Candidate probabilistic models:** Bayesian logistic regressions on many subsets of the code metrics (greedy BIC forward selection + **MC3** Metropolis search over subsets), plus Gaussian Naive Bayes.
* **Posterior model weights:** BIC (Schwarz) approximation to the marginal likelihood — a built-in Occam's razor.
* **Parameter uncertainty:** each logistic member has a Laplace (Gaussian) posterior over its coefficients, so each prediction is a full distribution and gets a **credible interval**.
* **Uncertainty split:** aleatoric (expected entropy) vs epistemic (mutual information); between-model vs within-model variance.

In [ ]:
R = B.run_study(df, seed=SEED)
bma, pre = R["bma"], R["pre"]
print(f"MC3 visited {bma.n_visited_} metric subsets; kept {len(bma.models_)} candidate models")
wt = bma.weights_table()
print("effective number of models (1/Σw²):", round(1/(bma.weights_**2).sum(), 1))
wt.head(10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
t = wt.head(14).iloc[::-1]
ax[0].barh(t.model, t.posterior_weight, color="#7c3aed"); ax[0].set_title("Posterior model weights  P(Mₖ | D)"); ax[0].set_xlabel("weight")
pip = bma.inclusion_probabilities().head(15).iloc[::-1]
ax[1].barh(pip.index, pip.values, color="#06b6d4"); ax[1].set_title("Posterior inclusion probability of each metric"); ax[1].set_xlim(0, 1)
plt.tight_layout(); plt.show()

# effect of the posterior temperature tau (1 = textbook BIC posterior; <1 flatter)
rows = []
for tau in [1.0, .5, .25, .1]:
    bma.set_temperature(tau); rows.append((tau, round(1/(bma.weights_**2).sum(), 1), round(bma.weights_.max(), 3)))
bma.set_temperature(1.0)
pd.DataFrame(rows, columns=["tau", "effective_models", "max_weight"])

## 4 · Defect-risk output & uncertainty plots

In [ ]:
S = B.score_modules(R, R["splits"]["Xte"], level=0.90)
S["actual"] = R["yte"]
thr = R["thr"]["Bayesian Model Averaging"]
print(f"decision threshold (max-F1 on validation): {thr:.3f}")
print(S.risk_tier.value_counts().to_string())
S.sort_values("p_defect", ascending=False).head(10).round(3)

In [ ]:
D = bma.predict_dist(R["Zte"], level=0.90, n_draws=2000, seed=1)
top = S.sort_values("p_defect", ascending=False).head(20).iloc[::-1]
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
col = top.risk_tier.map({"High": "#ec4899", "Review": "#f59e0b", "Low": "#06b6d4"})
ax[0].errorbar(top.p_defect, range(len(top)), xerr=[top.p_defect-top.ci_low, top.ci_high-top.p_defect], fmt="none", ecolor="#999", lw=2)
ax[0].scatter(top.p_defect, range(len(top)), c=col, s=60, zorder=3, edgecolor="k")
ax[0].axvline(thr, ls=":", c="k"); ax[0].set_yticks(range(len(top))); ax[0].set_yticklabels([f"#{i}" for i in top.index], fontsize=7)
ax[0].set_title("Top-20 riskiest modules with 90% credible interval"); ax[0].set_xlabel("P(defect)")

sc = ax[1].scatter(D["aleatoric"], D["epistemic"], c=D["mean"], cmap="plasma", s=18); plt.colorbar(sc, ax=ax[1], label="P(defect)")
ax[1].set_xlabel("aleatoric (expected entropy)"); ax[1].set_ylabel("epistemic (mutual information)"); ax[1].set_title("Uncertainty decomposition")

order = np.argsort(D["epistemic"]); fr = np.arange(1.0, .45, -.05); a, b_ = [], []
for f in fr:
    k = order[:max(int(round(f*len(order))), 10)]; yy, pp = R["yte"][k], D["mean"][k]
    a.append(roc_auc_score(yy, pp)); b_.append(brier_score_loss(yy, pp))
ax[2].plot(fr*100, a, "o-", c="#7c3aed", label="ROC-AUC"); ax[2].set_xlabel("% of modules auto-scored (most certain first)"); ax[2].set_ylabel("ROC-AUC"); ax[2].invert_xaxis()
ax2 = ax[2].twinx(); ax2.plot(fr*100, b_, "s-", c="#ec4899", label="Brier"); ax2.set_ylabel("Brier"); ax2.grid(False)
ax[2].set_title("Uncertainty-aware triage (selective prediction)")
plt.tight_layout(); plt.show()
print(f"mean variance split — between-model: {D['between_var'].mean():.4f} | within-model (parameters): {D['within_var'].mean():.4f}")

In [ ]:
# posterior predictive of one high-risk and one borderline module
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for a_, idx, nm in zip(ax, [S.p_defect.values.argmax(), (S.p_defect-thr).abs().values.argmin()], ["highest-risk module", "borderline module"]):
    a_.hist(D["draws"][idx], bins=40, color="#7c3aed", alpha=.85)
    for v, c in [(D["lo"][idx], "#999"), (D["hi"][idx], "#999"), (thr, "#f59e0b")]: a_.axvline(v, ls="--", c=c)
    a_.set_title(f"{nm}: P={D['mean'][idx]:.2f}  [{D['lo'][idx]:.2f}, {D['hi'][idx]:.2f}]"); a_.set_xlim(0, 1)
plt.tight_layout(); plt.show()

## 5 · Calibration
Probabilities must mean what they say. We report Brier score, log-loss and expected calibration error (ECE) and plot reliability diagrams.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
for nm in ["Bayesian Model Averaging", "Logistic Regression", "Random Forest", "Naive Bayes"]:
    px, py, _ = B.reliability_curve(R["yte"], R["probs"][nm][1], 6)
    ax[0].plot(px, py, "o-", lw=3 if nm.startswith("Bayes") else 1.6, label=nm)
ax[0].set_xlabel("mean predicted probability"); ax[0].set_ylabel("observed defect frequency"); ax[0].legend(fontsize=8); ax[0].set_title("Reliability diagram (test)")
pb = R["probs"]["Bayesian Model Averaging"][1]
ax[1].hist(pb[R["yte"] == 0], bins=25, alpha=.7, label="clean", color="#06b6d4"); ax[1].hist(pb[R["yte"] == 1], bins=25, alpha=.7, label="defective", color="#ec4899")
ax[1].legend(); ax[1].set_title("BMA predicted probabilities"); plt.tight_layout(); plt.show()
R["table"][["Algorithm", "Test Brier", "Test Log-loss", "Test ECE"]].round(4)

## 6 · Model-comparison table
Same held-out test set for every algorithm · hyper-parameters (LogReg *C*, RF depth/leaf, NB smoothing) and the F1 decision threshold tuned on **validation** data only · measured results, not assumed accuracy.

In [ ]:
T = R["table"].copy()
cols = ["Algorithm", "Validation ROC-AUC", "Test ROC-AUC", "Test F1", "Test Brier", "Test Log-loss", "Test ECE", "Training time (s)", "Inference time (ms/1000)", "Interpretability"]
T[cols].style.format({c: "{:.3f}" for c in cols[1:7]} | {"Training time (s)": "{:.3f}", "Inference time (ms/1000)": "{:.1f}"}) \
  .highlight_max(subset=["Validation ROC-AUC", "Test ROC-AUC", "Test F1"], color="#ddd6fe") \
  .highlight_min(subset=["Test Brier", "Test Log-loss", "Test ECE"], color="#ddd6fe")

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for nm in R["probs"]:
    p = R["probs"][nm][1]; f, t_, _ = roc_curve(R["yte"], p)
    ax[0].plot(f, t_, lw=3 if nm.startswith("Bayes") else 1.5, label=f"{nm} ({roc_auc_score(R['yte'], p):.3f})")
    pr, rc, _ = precision_recall_curve(R["yte"], p); ax[1].plot(rc, pr, lw=3 if nm.startswith("Bayes") else 1.5, label=f"{nm} ({average_precision_score(R['yte'], p):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set_title("ROC (test)"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend(fontsize=7)
ax[1].set_title("Precision-recall (test)"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 7 · Robustness
A single split of a small NASA dataset is noisy (the test set has only a few dozen defective modules). We therefore repeat the **entire** pipeline on 10 random splits, then on **all nine** NASA datasets.

In [ ]:
rep = B.repeated_study(df, seeds=range(10))
display(B.summarise_repeated(rep))
fig, ax = plt.subplots(figsize=(8, 3.6))
order = ["Bayesian Model Averaging", "Best single BIC model", "Logistic Regression", "Naive Bayes", "Random Forest"]
ax.boxplot([rep[rep.Algorithm == a]["Test ROC-AUC"] for a in order], patch_artist=True, boxprops=dict(facecolor="#ddd6fe"), medianprops=dict(color="#7c3aed"))
ax.set_xticklabels(["BMA", "Best-BIC", "LogReg", "NB", "RF"]); ax.set_ylabel("Test ROC-AUC"); ax.set_title(f"{DATASET}: 10 random splits"); plt.show()

In [ ]:
# all NASA datasets, 5 splits each (takes ~2-3 minutes)
rows = []
for name in B.DATASETS:
    d = B.load_dataset(name)
    r = B.repeated_study(d, seeds=range(5))
    g = r.groupby("Algorithm")[["Test ROC-AUC", "Test Brier", "Test ECE"]].mean()
    for a in g.index: rows.append(dict(dataset=name, algorithm=a, **g.loc[a].to_dict()))
big = pd.DataFrame(rows)
piv = big.pivot(index="dataset", columns="algorithm", values="Test ROC-AUC")[order]
display(piv.round(3).style.highlight_max(axis=1, color="#ddd6fe").set_caption("Mean test ROC-AUC over 5 splits"))
display(big.pivot(index="dataset", columns="algorithm", values="Test Brier")[order].round(3).style.highlight_min(axis=1, color="#ddd6fe").set_caption("Mean test Brier (lower = better)"))
print("Mean over datasets:"); print(big.groupby("algorithm")[["Test ROC-AUC", "Test Brier", "Test ECE"]].mean().loc[order].round(3))

## 8 · Conclusions
* **Measured result (not assumed):** averaged over all nine NASA datasets, BMA matches Random Forest on ROC-AUC (≈0.75), beats the best single BIC model and Naive Bayes, and is within ~0.02 AUC of a tuned Logistic Regression, with similarly low Brier/ECE. It is *not* uniformly the most accurate — its edge is that it **also returns** a posterior over predictions**: credible intervals, epistemic vs. aleatoric uncertainty and posterior inclusion probabilities of each code metric — none of which a single logistic regression or a random forest provides out of the box.
* **Uncertainty is actionable:** modules whose credible interval straddles the decision threshold are routed to human review; auto-scoring only the most certain modules measurably improves AUC/Brier (selective-prediction plot).
* **Honest limits.** On several NASA datasets all methods are statistically close (see the ± across 10 splits); Naive Bayes is badly calibrated (independence assumption on strongly correlated metrics); BIC weights concentrate on few models when *n* is large (JM1), shrinking model uncertainty — the temperature τ exposes that trade-off. NASA MDP has no timestamps/entity IDs, so a stratified random split is used and may be optimistic versus a time-ordered deployment.
* **Extensions** (syllabus): boosting / GAM / CART members in the candidate set, hierarchical BMA across projects, MCMC instead of Laplace.

## 9 · 🚀 Web demo — DefectLens (Streamlit, glassmorphism UI)
This writes the app, then opens it through a free Cloudflare tunnel so you get a public link from inside Colab. Keep this cell running while you demo.

In [ ]:
%%writefile app.py
"""
DefectLens - Bayesian Model Averaging for software defect prediction
Glassmorphism Streamlit demo.      Run:  streamlit run app.py
"""
import hashlib
import io
import os
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
from sklearn.metrics import (average_precision_score, brier_score_loss, precision_recall_curve,
                             roc_auc_score, roc_curve)

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import bma as B  # noqa: E402

st.set_page_config(page_title="DefectLens | Bayesian defect prediction", page_icon="🔮",
                   layout="wide", initial_sidebar_state="expanded")

# =============================================================================
# THEME  (dark violet glassmorphism, inspired by the Reflect landing page)
# =============================================================================
VIOLET, LILAC, CYAN, PINK, AMBER, GREEN = "#8b5cf6", "#c4b5fd", "#22d3ee", "#f472b6", "#fbbf24", "#34d399"
TIER_COLOR = {"High": PINK, "Review": AMBER, "Low": CYAN}

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap');
html, body, .stApp, button, input, textarea, .stMarkdown, .stMarkdown *, [data-testid="stTab"] *, .stSlider *, label, .stSelectbox * { font-family: 'Inter', 'Segoe UI', 'Helvetica Neue', Arial, sans-serif !important; }
.stApp {
  background:
    radial-gradient(1100px 520px at 50% -8%, rgba(99,52,214,.30), transparent 60%),
    radial-gradient(700px 420px at 90% 38%, rgba(34,211,238,.07), transparent 65%),
    radial-gradient(800px 500px at 8% 62%, rgba(168,85,247,.12), transparent 65%),
    linear-gradient(180deg, #05030f 0%, #070416 45%, #04020c 100%);
  color: #e9e5ff;
}
#MainMenu, footer, header[data-testid="stHeader"], [data-testid="stToolbar"], [data-testid="stDecoration"] { display:none !important; }
.block-container { max-width: 1260px; padding-top: 1.1rem; padding-bottom: 4rem; }
h1,h2,h3,h4 { color:#f5f3ff; letter-spacing:-.02em; }
p, li, label, span, div { color: inherit; }
hr { border-color: rgba(255,255,255,.08) !important; }

/* ---------- glass primitives ---------- */
.glass, div[data-testid="stVerticalBlock"]:has(> [data-testid="stElementContainer"] .card-title) {
  background: linear-gradient(180deg, rgba(255,255,255,.07), rgba(255,255,255,.022)) !important;
  border: 1px solid rgba(255,255,255,.11) !important; border-radius: 22px !important;
  backdrop-filter: blur(22px) saturate(140%); -webkit-backdrop-filter: blur(22px) saturate(140%);
  box-shadow: 0 10px 50px rgba(76,29,200,.16), inset 0 1px 0 rgba(255,255,255,.10);
  padding: 14px 16px 10px !important;
}
.card-title { font-weight:600; font-size:1.02rem; color:#f5f3ff; margin:.2rem 0 .1rem; letter-spacing:-.01em; }
.card-sub { font-size:.82rem; color:#a79fd0; margin-bottom:.4rem; line-height:1.45; }

/* ---------- top nav ---------- */
.nav { display:flex; align-items:center; justify-content:space-between; gap:16px; margin-bottom:6px; }
.brand { display:flex; align-items:center; gap:11px; font-weight:700; font-size:1.08rem; color:#fff; letter-spacing:-.02em; }
.logo { width:38px; height:38px; border-radius:11px; background: radial-gradient(circle at 30% 25%, #a78bfa, #4c1d95 70%);
  box-shadow: 0 0 22px rgba(139,92,246,.65), inset 0 0 0 1px rgba(255,255,255,.25); display:grid; place-items:center; }
.logo i { width:18px; height:18px; border-radius:50%; border:2px solid rgba(255,255,255,.9); box-shadow:0 0 10px rgba(255,255,255,.7) inset; }
.navpill { padding:9px 22px; border-radius:999px; background:rgba(255,255,255,.05); border:1px solid rgba(255,255,255,.10);
  backdrop-filter: blur(18px); font-size:.86rem; color:#cfc8ee; display:flex; gap:18px; align-items:center; }
.navpill b { color:#fff; font-weight:600; }
.dot { width:8px; height:8px; border-radius:50%; background:#34d399; box-shadow:0 0 10px #34d399; display:inline-block; margin-right:7px; }
.cta { padding:10px 20px; border-radius:12px; background:linear-gradient(180deg, rgba(139,92,246,.35), rgba(91,33,182,.35));
  border:1px solid rgba(167,139,250,.45); color:#fff; font-weight:600; font-size:.87rem; box-shadow:0 0 24px rgba(139,92,246,.35); }

/* ---------- hero ---------- */
.hero { text-align:center; padding: 40px 0 0; position:relative; }
.badge { display:inline-flex; gap:9px; align-items:center; padding:8px 16px; border-radius:999px; font-size:.82rem; color:#d9d3ff;
  background:rgba(255,255,255,.05); border:1px solid rgba(255,255,255,.11); backdrop-filter: blur(14px); }
.hero h1 { font-size: clamp(2.4rem, 5.4vw, 4.4rem); font-weight:800; line-height:1.04; margin:22px 0 14px;
  background: linear-gradient(180deg, #ffffff 25%, #b9acf5 100%); -webkit-background-clip:text; background-clip:text; color:transparent; }
.hero p.sub { color:#b7aedf; font-size:1.08rem; margin:0 auto; max-width:640px; }
.halo { position:relative; height:250px; margin-top:6px; overflow:hidden; }
.halo .glow { position:absolute; left:50%; bottom:-170px; width:980px; height:420px; transform:translateX(-50%);
  background: radial-gradient(ellipse at 50% 100%, rgba(170,140,255,.85) 0%, rgba(124,58,237,.45) 28%, rgba(76,29,149,.18) 52%, transparent 72%);
  filter: blur(26px); animation: breathe 7s ease-in-out infinite; }
.halo .arc { position:absolute; left:50%; bottom:-2px; transform:translateX(-50%); border-radius: 50% 50% 0 0 / 100% 100% 0 0; border-bottom:none; }
.halo .a3 { width:640px; height:215px; border:2px solid rgba(167,139,250,.28); box-shadow:0 0 50px rgba(139,92,246,.4); }
.halo .a2 { width:470px; height:160px; border:4px solid rgba(221,214,254,.75); box-shadow:0 0 36px 4px rgba(167,139,250,.85), inset 0 0 30px rgba(167,139,250,.55); }
.halo .a1 { width:320px; height:108px; border:5px solid #fff; box-shadow:0 0 30px 6px rgba(255,255,255,.55), 0 0 70px 14px rgba(167,139,250,.9), inset 0 0 30px rgba(196,181,253,.9);
  background: radial-gradient(ellipse at 50% 100%, rgba(124,58,237,.9), rgba(30,10,80,.9) 70%); }
.halo .line { position:absolute; left:6%; right:6%; bottom:0; height:2px; background:linear-gradient(90deg, transparent, rgba(196,181,253,.9), transparent); box-shadow:0 0 22px 4px rgba(167,139,250,.7); }
@keyframes breathe { 0%,100% { opacity:.85; transform:translateX(-50%) scale(1);} 50% { opacity:1; transform:translateX(-50%) scale(1.06);} }

/* ---------- KPI bar ---------- */
.kpibar { display:grid; grid-template-columns: repeat(5, 1fr); gap:0; border-radius:20px; padding:16px 8px; margin: -34px 0 26px;
  background: linear-gradient(180deg, rgba(255,255,255,.085), rgba(255,255,255,.03)); border:1px solid rgba(255,255,255,.12);
  backdrop-filter: blur(24px); box-shadow: 0 18px 60px rgba(76,29,200,.30), inset 0 1px 0 rgba(255,255,255,.12); position:relative; z-index:2; }
.kpi { text-align:center; padding: 4px 8px; border-right:1px solid rgba(255,255,255,.07); }
.kpi:last-child { border-right:none; }
.kpi .v { font-size:1.65rem; font-weight:700; color:#fff; letter-spacing:-.02em; }
.kpi .l { font-size:.74rem; text-transform:uppercase; letter-spacing:.09em; color:#9f96cc; margin-top:2px; }
.kpi .d { font-size:.74rem; color:#34d399; }

/* ---------- stat cards ---------- */
.stat { padding:18px 20px; border-radius:20px; background:linear-gradient(180deg, rgba(255,255,255,.07), rgba(255,255,255,.025));
  border:1px solid rgba(255,255,255,.10); backdrop-filter: blur(20px); box-shadow:0 8px 40px rgba(76,29,200,.12), inset 0 1px 0 rgba(255,255,255,.09); }
.stat .l { font-size:.76rem; color:#a79fd0; text-transform:uppercase; letter-spacing:.08em; }
.stat .v { font-size:2.0rem; font-weight:700; color:#fff; letter-spacing:-.02em; margin-top:4px; }
.stat .s { font-size:.8rem; color:#8f86bd; }
.pill { display:inline-block; padding:3px 11px; border-radius:999px; font-size:.74rem; font-weight:600; }
.p-High { background:rgba(244,114,182,.16); color:#f9a8d4; border:1px solid rgba(244,114,182,.35); }
.p-Review { background:rgba(251,191,36,.14); color:#fcd34d; border:1px solid rgba(251,191,36,.35); }
.p-Low { background:rgba(34,211,238,.13); color:#67e8f9; border:1px solid rgba(34,211,238,.32); }

/* ---------- tables ---------- */
.gt-wrap { overflow-x:auto; border-radius:16px; border:1px solid rgba(255,255,255,.08); background:rgba(8,5,24,.35); }
table.gt { width:100%; border-collapse:collapse; font-size:.84rem; }
table.gt th { text-align:left; padding:11px 14px; color:#b4abe0; font-weight:600; font-size:.74rem; text-transform:uppercase; letter-spacing:.07em;
  background:rgba(255,255,255,.04); border-bottom:1px solid rgba(255,255,255,.08); white-space:nowrap; }
table.gt td { padding:10px 14px; border-bottom:1px solid rgba(255,255,255,.05); color:#e7e3ff; white-space:nowrap; }
table.gt tr:hover td { background:rgba(139,92,246,.09); }
table.gt td.best { color:#fff; font-weight:700; background:rgba(139,92,246,.17); }
table.gt td.wrap { white-space:normal; min-width:240px; color:#b7aedf; }

/* ---------- streamlit widgets ---------- */
section[data-testid="stSidebar"] { background: linear-gradient(180deg, rgba(14,8,40,.92), rgba(6,3,18,.95)); border-right:1px solid rgba(255,255,255,.08);
  backdrop-filter: blur(24px); }
section[data-testid="stSidebar"] * { color:#ddd8f7; }
[data-testid="stTabs"] [role="tablist"] { gap:2px; background:rgba(255,255,255,.05); border:1px solid rgba(255,255,255,.11); border-radius:999px;
  padding:6px; width:fit-content; margin:0 auto 24px; backdrop-filter: blur(20px); flex-wrap:wrap; justify-content:center; box-shadow:0 8px 40px rgba(76,29,200,.2); }
[data-testid="stTabs"] [role="tablist"] ~ * , [data-testid="stTabs"] > div:first-child { border-bottom:none !important; }
[data-testid="stTab"] { border-radius:999px; padding:9px 15px; height:auto; color:#b4abe0; font-weight:500; background:transparent; border:none; }
[data-testid="stTab"] p { color:inherit; font-size:.92rem; margin:0; }
[data-testid="stTab"][aria-selected="true"] { background:linear-gradient(180deg, rgba(139,92,246,.6), rgba(91,33,182,.55)) !important; color:#fff !important;
  box-shadow:0 0 22px rgba(139,92,246,.55), inset 0 1px 0 rgba(255,255,255,.28); }
.react-aria-SelectionIndicator { display:none !important; }
[data-testid="stTabs"] [role="tablist"]::after, [data-testid="stTabs"] [role="tablist"]::before { display:none !important; }
.stButton>button, .stDownloadButton>button { border-radius:12px; border:1px solid rgba(167,139,250,.45); color:#fff; font-weight:600;
  background:linear-gradient(180deg, rgba(139,92,246,.40), rgba(91,33,182,.38)); box-shadow:0 0 22px rgba(139,92,246,.3); }
.stButton>button:hover, .stDownloadButton>button:hover { border-color:#c4b5fd; box-shadow:0 0 30px rgba(167,139,250,.6); color:#fff; }
div[data-baseweb="select"]>div, .stTextInput input, .stNumberInput input { background:rgba(255,255,255,.05) !important; border:1px solid rgba(255,255,255,.12) !important; border-radius:12px !important; color:#fff !important; }
[data-testid="stSlider"] [role="slider"] { background:#a78bfa !important; box-shadow:0 0 14px #8b5cf6; }
[data-testid="stFileUploaderDropzone"] { background:rgba(255,255,255,.04); border:1px dashed rgba(167,139,250,.4); border-radius:14px; }
[data-testid="stMetric"] { background:rgba(255,255,255,.04); border-radius:14px; padding:10px 14px; border:1px solid rgba(255,255,255,.08); }
.stRadio [role="radiogroup"] { gap:8px; }
.footer { text-align:center; color:#6f67a0; font-size:.8rem; margin-top:34px; }
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)


# =============================================================================
# HELPERS
# =============================================================================
def H(html: str):
    st.markdown("\n".join(l.strip() for l in html.strip().splitlines()), unsafe_allow_html=True)


def card_head(title, sub=""):
    H(f'<div class="card-title">{title}</div><div class="card-sub">{sub}</div>')


def style_fig(fig, h=360, legend=True, **kw):
    fig.update_layout(
        height=h, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        font=dict(family="Inter", color="#cfc8ee", size=12), margin=dict(l=10, r=10, t=30, b=10),
        colorway=[VIOLET, CYAN, PINK, AMBER, GREEN, LILAC],
        legend=dict(bgcolor="rgba(255,255,255,.04)", bordercolor="rgba(255,255,255,.08)", borderwidth=1,
                    font=dict(size=11), orientation="h", y=1.14, x=0) if legend else dict(visible=False),
        hoverlabel=dict(bgcolor="#1b1140", bordercolor="#8b5cf6", font=dict(color="#fff")), **kw)
    fig.update_xaxes(gridcolor="rgba(255,255,255,.06)", zerolinecolor="rgba(255,255,255,.10)", linecolor="rgba(255,255,255,.12)")
    fig.update_yaxes(gridcolor="rgba(255,255,255,.06)", zerolinecolor="rgba(255,255,255,.10)", linecolor="rgba(255,255,255,.12)")
    return fig


def plot(fig, key=None):
    st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False}, key=key)


def gtable(df, fmt=None, best=None, wrap=(), raw=()):
    """Glass HTML table. best = {col: 'max'|'min'} highlights the winner."""
    fmt = fmt or {}
    best = best or {}
    win = {c: (df[c].idxmax() if m == "max" else df[c].idxmin()) for c, m in best.items() if c in df}
    head = "".join(f"<th>{c}</th>" for c in df.columns)
    rows = []
    for i, r in df.iterrows():
        tds = []
        for c in df.columns:
            v = r[c]
            if c in raw:
                s = v
            elif c in fmt and isinstance(v, (int, float, np.floating, np.integer)):
                s = fmt[c].format(v)
            else:
                s = v
            cls = "best" if win.get(c) == i else ("wrap" if c in wrap else "")
            tds.append(f'<td class="{cls}">{s}</td>')
        rows.append("<tr>" + "".join(tds) + "</tr>")
    H(f'<div class="gt-wrap"><table class="gt"><thead><tr>{head}</tr></thead><tbody>{"".join(rows)}</tbody></table></div>')


def pill(t):
    return f'<span class="pill p-{t}">{t}</span>'


# =============================================================================
# DATA + MODEL (cached)
# =============================================================================
def to_binary(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.astype(int)
    if pd.api.types.is_numeric_dtype(s):
        return (s > 0).astype(int)
    return s.astype(str).str.strip().str.lower().isin(["y", "yes", "true", "1", "defective", "buggy"]).astype(int)


@st.cache_data(show_spinner=False)
def get_dataset(name):
    return B.load_dataset(name)


@st.cache_resource(show_spinner=False)
def get_study(key, df_bytes, seed, test_size, tau, max_features):
    df = pd.read_pickle(io.BytesIO(df_bytes))
    return B.run_study(df, seed=seed, test_size=test_size, val_size=0.2, temperature=tau,
                       max_features=max_features, n_draws=2000)


@st.cache_data(show_spinner=False)
def get_repeated(key, df_bytes, n, tau):
    df = pd.read_pickle(io.BytesIO(df_bytes))
    rep = B.repeated_study(df, seeds=range(n), temperature=tau)
    return rep


# =============================================================================
# SIDEBAR
# =============================================================================
with st.sidebar:
    H('<div class="brand" style="margin-bottom:14px"><div class="logo"><i></i></div>DefectLens</div>')
    st.caption("Bayesian Model Averaging · software defect prediction")
    src = st.radio("Data source", ["NASA / PROMISE dataset", "Upload my CSV"], horizontal=False)
    if src.startswith("NASA"):
        ds = st.selectbox("Dataset", list(B.DATASETS), index=3, format_func=lambda k: f"{k} · {B.DATASETS[k].split(' - ')[1]}")
        try:
            raw = get_dataset(ds)
        except Exception as e:  # noqa: BLE001
            st.error(f"Could not load {ds}: {e}")
            st.stop()
        data_key = ds
    else:
        up = st.file_uploader("CSV with code metrics + a binary defect column", type="csv")
        if up is None:
            st.info("Upload a CSV, or switch back to a NASA dataset.")
            st.stop()
        udf = pd.read_csv(up)
        tgt = st.selectbox("Defect column", [c for c in udf.columns[::-1]])
        feats = udf.drop(columns=[tgt]).select_dtypes("number")
        raw = feats.copy()
        raw[B.TARGET] = to_binary(udf[tgt]).values
        data_key = "upload-" + hashlib.md5(pd.util.hash_pandas_object(raw).values.tobytes()).hexdigest()[:10]
    st.markdown("---")
    seed = st.number_input("Random seed (split)", 0, 9999, 42, 1)
    test_size = st.slider("Held-out test share", 0.15, 0.40, 0.20, 0.05)
    tau = st.slider("Posterior temperature τ", 0.10, 1.00, 1.00, 0.05,
                    help="τ=1 is the textbook BIC posterior. Lower τ flattens the model weights -> more model uncertainty.")
    max_f = st.slider("Max metrics per model", 3, 14, 10)
    level = st.select_slider("Credible interval", [0.80, 0.90, 0.95], value=0.90, format_func=lambda v: f"{int(v*100)}%")
    st.markdown("---")
    st.caption("Same held-out test set for every algorithm · preprocessing fitted on training data only · "
               "thresholds & hyper-parameters tuned on validation data only.")

buf = io.BytesIO()
raw.to_pickle(buf)
with st.spinner("Fitting Bayesian candidate models and baselines …"):
    R = get_study(data_key, buf.getvalue(), int(seed), float(test_size), float(tau), int(max_f))

bma, pre, sp, tab = R["bma"], R["pre"], R["splits"], R["table"]
thr = R["thr"]["Bayesian Model Averaging"]
row = tab.set_index("Algorithm")
b = row.loc["Bayesian Model Averaging"]
w = bma.weights_
n_eff = 1.0 / np.sum(w ** 2)
n_all = sp["n_dedup"]
rate = (np.concatenate([sp["ytr"], sp["yva"], sp["yte"]])).mean()

# =============================================================================
# NAV + HERO
# =============================================================================
H(f"""
<div class="nav">
  <div class="brand"><div class="logo"><i></i></div>DefectLens</div>
  <div class="navpill"><span><span class="dot"></span><b>{data_key if not data_key.startswith('upload') else 'Your data'}</b></span>
    <span>{n_all:,} unique modules</span><span>{rate*100:.1f}% defective</span><span>{len(pre.columns_)} metrics kept</span></div>
  <div class="cta">Bayesian Model Averaging</div>
</div>
<div class="hero">
  <div class="badge"><span style="color:#c4b5fd">✦</span> New: posterior model weights &amp; credible intervals, live</div>
  <h1>Predict defects.<br>Quantify the doubt.</h1>
  <p class="sub">Never ship a risky module blind. DefectLens averages hundreds of Bayesian models and tells you
  not just <i>what</i> is likely to fail — but how sure it is.</p>
  <div class="halo"><div class="glow"></div><div class="arc a3"></div><div class="arc a2"></div><div class="arc a1"></div><div class="line"></div></div>
</div>
<div class="kpibar">
  <div class="kpi"><div class="v">{b['Test ROC-AUC']:.3f}</div><div class="l">Test ROC-AUC</div></div>
  <div class="kpi"><div class="v">{b['Test F1']:.3f}</div><div class="l">Test F1</div></div>
  <div class="kpi"><div class="v">{b['Test Brier']:.3f}</div><div class="l">Brier score</div></div>
  <div class="kpi"><div class="v">{b['Test ECE']:.3f}</div><div class="l">Calibration error</div></div>
  <div class="kpi"><div class="v">{n_eff:.1f}</div><div class="l">Effective models</div></div>
</div>
""")

# scored modules (held-out test set by default)
scope = "Held-out test set"
Xte_raw = sp["Xte"]
S_test = B.score_modules(R, Xte_raw, level=level, n_draws=1000)
dist_test = bma.predict_dist(R["Zte"], level=level, n_draws=1000, seed=7)

tabs = st.tabs(["◉ Risk dashboard", "⚖ Model weights", "≈ Uncertainty", "◐ Calibration", "▤ Comparison", "⚡ Playground", "ⓘ About"])

# -----------------------------------------------------------------------------
# TAB 1 - RISK DASHBOARD
# -----------------------------------------------------------------------------
with tabs[0]:
    c1, c2, c3, c4 = st.columns(4)
    cnt = S_test.risk_tier.value_counts()
    for col, label, sub, val in [
        (c1, "High risk", "credible interval fully above threshold", cnt.get("High", 0)),
        (c2, "Needs review", "interval straddles the threshold", cnt.get("Review", 0)),
        (c3, "Low risk", "interval fully below threshold", cnt.get("Low", 0)),
        (c4, f"Avg {int(level*100)}% interval width", "mean predictive uncertainty", f"{S_test.ci_width.mean():.2f}")]:
        col.markdown(f'<div class="stat"><div class="l">{label}</div><div class="v">{val}</div><div class="s">{sub}</div></div>', unsafe_allow_html=True)
    st.write("")
    left, right = st.columns([1.15, 1])
    with left:
        with st.container(border=True):
            card_head("Top-20 riskiest modules", f"Posterior mean P(defect) with {int(level*100)}% credible interval · ◯ = actually defective")
            top = S_test.sort_values("p_defect", ascending=False).head(20).iloc[::-1]
            lab = [f"module #{i}" for i in top.index]
            yv = pd.Series(sp["yte"], index=Xte_raw.index).loc[top.index].values
            fig = go.Figure()
            for t in ["High", "Review", "Low"]:
                m = (top.risk_tier == t).values
                if m.any():
                    fig.add_trace(go.Scatter(x=top.p_defect[m], y=np.array(lab)[m], mode="markers", name=f"{t}",
                        marker=dict(color=TIER_COLOR[t], size=10, line=dict(width=1, color="#fff")),
                        error_x=dict(type="data", symmetric=False, array=(top.ci_high - top.p_defect)[m],
                                     arrayminus=(top.p_defect - top.ci_low)[m], color=TIER_COLOR[t], thickness=2.2, width=0),
                        hovertemplate="%{y}<br>P(defect)=%{x:.2f}<extra></extra>"))
            fig.add_trace(go.Scatter(x=top.p_defect[yv == 1], y=np.array(lab)[yv == 1], mode="markers", name="actual defect",
                                     marker=dict(symbol="circle-open", size=19, color="#fff", line=dict(width=1.6)), hoverinfo="skip"))
            fig.add_vline(x=thr, line_dash="dot", line_color=LILAC, annotation_text=f"threshold {thr:.2f}", annotation_font_color=LILAC)
            style_fig(fig, 520)
            fig.update_xaxes(range=[0, 1.02], title="P(defect)")
            plot(fig, "top20")
    with right:
        with st.container(border=True):
            card_head("Risk vs. uncertainty map", "Each dot is a held-out module — high & wide = risky and unsure")
            fig = go.Figure()
            for t in ["High", "Review", "Low"]:
                m = S_test.risk_tier == t
                fig.add_trace(go.Scatter(x=S_test.p_defect[m], y=S_test.ci_width[m], mode="markers", name=t,
                    marker=dict(color=TIER_COLOR[t], size=7, opacity=.75, line=dict(width=0)),
                    hovertemplate="P=%{x:.2f}<br>width=%{y:.2f}<extra></extra>"))
            fig.add_vline(x=thr, line_dash="dot", line_color=LILAC)
            style_fig(fig, 250)
            fig.update_xaxes(title="P(defect)", range=[0, 1])
            fig.update_yaxes(title="interval width")
            plot(fig, "riskmap")
            card_head("Risk-tier mix", "")
            tiers = S_test.risk_tier.value_counts().reindex(["High", "Review", "Low"]).fillna(0)
            fig = go.Figure(go.Bar(x=tiers.values, y=tiers.index, orientation="h", marker_color=[TIER_COLOR[t] for t in tiers.index],
                                   text=tiers.values.astype(int), textposition="outside", textfont=dict(color="#fff")))
            style_fig(fig, 150, legend=False)
            fig.update_xaxes(range=[0, tiers.max() * 1.2])
            plot(fig, "tiers")
    with st.container(border=True):
        card_head("Module risk table", "Sorted by posterior defect probability — download the full scored list below")
        show = S_test.copy()
        show["actual"] = np.where(pd.Series(sp["yte"], index=Xte_raw.index).loc[show.index] == 1, "defective", "clean")
        show = show.sort_values("p_defect", ascending=False)
        keymetrics = [c for c in ["LOC_TOTAL", "CYCLOMATIC_COMPLEXITY"] if c in Xte_raw.columns] or list(pre.columns_[:2])
        t = pd.DataFrame({"Module": [f"#{i}" for i in show.index[:15]],
                          "P(defect)": [f"{v*100:.1f}%" for v in show.p_defect[:15]],
                          f"{int(level*100)}% interval": [f"{a*100:.0f}% – {c*100:.0f}%" for a, c in zip(show.ci_low[:15], show.ci_high[:15])],
                          "Tier": [pill(x) for x in show.risk_tier[:15]],
                          "Epistemic": [f"{v:.3f}" for v in show.epistemic_uncertainty[:15]],
                          "Truth": show.actual[:15].values})
        for k in keymetrics:
            t[k] = Xte_raw.loc[show.index[:15], k].round(1).values
        gtable(t, raw=("Tier",))
        out = show.join(Xte_raw)
        st.download_button("⬇ Download scored modules (CSV)", out.to_csv().encode(), "defectlens_scored_modules.csv", "text/csv")

# -----------------------------------------------------------------------------
# TAB 2 - MODEL WEIGHTS
# -----------------------------------------------------------------------------
with tabs[1]:
    wt = bma.weights_table()
    c1, c2, c3, c4 = st.columns(4)
    for col, label, val, sub in [(c1, "Candidate models", len(bma.models_), f"{bma.n_visited_} subsets visited by MC3"),
                                 (c2, "Effective models", f"{n_eff:.1f}", "1 / Σ weight²"),
                                 (c3, "Top model weight", f"{w.max()*100:.1f}%", bma.best_model_["name"]),
                                 (c4, "Occam's window", f"{(w>0.01).sum()}", "models with weight > 1%")]:
        col.markdown(f'<div class="stat"><div class="l">{label}</div><div class="v">{val}</div><div class="s">{sub}</div></div>', unsafe_allow_html=True)
    st.write("")
    l, r = st.columns(2)
    with l:
        with st.container(border=True):
            card_head("Posterior model weights  P(Mₖ | data)", "∝ exp(−τ·BIC/2) — Schwarz approximation to the marginal likelihood")
            top = wt.head(14).iloc[::-1]
            colr = {"logreg": VIOLET, "nb": CYAN, "tree": PINK}
            fig = go.Figure(go.Bar(x=top.posterior_weight, y=top.model, orientation="h", marker_color=[colr[k] for k in top.kind],
                                   customdata=np.c_[top.params, top.BIC.round(1)],
                                   hovertemplate="%{y}<br>weight %{x:.3f}<br>params %{customdata[0]} · BIC %{customdata[1]}<extra></extra>"))
            style_fig(fig, 430, legend=False)
            fig.update_xaxes(title="posterior weight")
            plot(fig, "weights")
    with r:
        with st.container(border=True):
            card_head("Posterior inclusion probability of each metric", "Share of posterior mass on models that include the metric")
            pip = bma.inclusion_probabilities().head(15).iloc[::-1]
            fig = go.Figure(go.Bar(x=pip.values, y=pip.index, orientation="h",
                                   marker=dict(color=pip.values, colorscale=[[0, "#3b1d8f"], [1, "#e9d5ff"]], line=dict(width=0))))
            style_fig(fig, 430, legend=False)
            fig.update_xaxes(range=[0, 1.02], title="inclusion probability")
            plot(fig, "pip")
    with st.container(border=True):
        card_head("Model leaderboard", "Complexity penalty comes from BIC = −2·logL + k·ln n")
        lt = wt.head(10).copy()
        lt["posterior_weight"] = (lt.posterior_weight * 100).map("{:.2f}%".format)
        lt["log_likelihood"] = lt.log_likelihood.map("{:.1f}".format)
        lt["BIC"] = lt.BIC.map("{:.1f}".format)
        lt["metrics"] = lt.metrics.map(lambda s: (s[:70] + "…") if len(s) > 70 else s)
        lt = lt.rename(columns={"model": "Model", "kind": "Family", "params": "k", "log_likelihood": "log-lik", "posterior_weight": "Weight", "metrics": "Metrics in model"})
        gtable(lt, wrap=("Metrics in model",))

# -----------------------------------------------------------------------------
# TAB 3 - UNCERTAINTY
# -----------------------------------------------------------------------------
with tabs[2]:
    yte = R["yte"]
    l, r = st.columns(2)
    with l:
        with st.container(border=True):
            card_head("Aleatoric vs. epistemic uncertainty", "Epistemic = what more data / better models could reduce (mutual information)")
            fig = go.Figure(go.Scatter(x=dist_test["aleatoric"], y=dist_test["epistemic"], mode="markers",
                                       marker=dict(size=7, color=dist_test["mean"], colorscale=[[0, CYAN], [.5, VIOLET], [1, PINK]],
                                                   showscale=True, colorbar=dict(title="P(defect)", thickness=10), opacity=.8, line=dict(width=0)),
                                       hovertemplate="aleatoric %{x:.3f}<br>epistemic %{y:.3f}<extra></extra>"))
            style_fig(fig, 360, legend=False)
            fig.update_xaxes(title="aleatoric (expected entropy, nats)")
            fig.update_yaxes(title="epistemic (mutual information)")
            plot(fig, "unc1")
    with r:
        with st.container(border=True):
            card_head("Where does the variance come from?", "Between-model (which model?) vs. within-model (which coefficients?)")
            bv, wv = dist_test["between_var"].mean(), dist_test["within_var"].mean()
            fig = go.Figure(go.Bar(x=["between-model", "within-model (parameters)"], y=[bv, wv], marker_color=[VIOLET, CYAN],
                                   text=[f"{bv:.4f}", f"{wv:.4f}"], textposition="outside", textfont=dict(color="#fff")))
            style_fig(fig, 360, legend=False)
            fig.update_yaxes(title="mean predictive variance")
            plot(fig, "unc2")
    with st.container(border=True):
        card_head("Uncertainty-aware triage (selective prediction)",
                  "Rank held-out modules by epistemic uncertainty, hand the most uncertain ones to a human reviewer, and score the rest")
        order = np.argsort(dist_test["epistemic"])
        fr = np.arange(1.0, 0.45, -0.05)
        auc_c, br_c = [], []
        for f in fr:
            keep = order[: max(int(round(f * len(order))), 10)]
            yy, pp = yte[keep], dist_test["mean"][keep]
            auc_c.append(roc_auc_score(yy, pp) if len(np.unique(yy)) > 1 else np.nan)
            br_c.append(brier_score_loss(yy, pp))
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=fr * 100, y=auc_c, mode="lines+markers", name="ROC-AUC (higher is better)", line=dict(color=VIOLET, width=3), marker=dict(size=8)))
        fig.add_trace(go.Scatter(x=fr * 100, y=br_c, mode="lines+markers", name="Brier (lower is better)", line=dict(color=PINK, width=3), marker=dict(size=8), yaxis="y2"))
        style_fig(fig, 340)
        fig.update_layout(yaxis2=dict(overlaying="y", side="right", gridcolor="rgba(0,0,0,0)", title="Brier"))
        fig.update_xaxes(title="% of modules auto-scored (most certain first)", autorange="reversed")
        fig.update_yaxes(title="ROC-AUC")
        plot(fig, "triage")
    with st.container(border=True):
        card_head("Posterior predictive of a single module", "The full distribution of P(defect) across models and parameter draws")
        order_p = S_test.sort_values("p_defect", ascending=False).index.tolist()
        pick = st.selectbox("Module", order_p[:60], format_func=lambda i: f"#{i} · P={S_test.p_defect[i]*100:.1f}% · {S_test.risk_tier[i]}")
        j = Xte_raw.index.get_loc(pick)
        dr = dist_test["draws"][j]
        fig = go.Figure(go.Histogram(x=dr, nbinsx=40, marker=dict(color=VIOLET, line=dict(width=0)), opacity=.85))
        for xv, nm, cl in [(dist_test["mean"][j], "mean", "#fff"), (dist_test["lo"][j], "lower", LILAC), (dist_test["hi"][j], "upper", LILAC), (thr, "decision threshold", AMBER)]:
            fig.add_vline(x=xv, line_dash="dash" if nm != "mean" else "solid", line_color=cl, annotation_text=nm, annotation_font_color=cl)
        style_fig(fig, 300, legend=False)
        fig.update_xaxes(title="P(defect)", range=[0, 1])
        plot(fig, "single")

# -----------------------------------------------------------------------------
# TAB 4 - CALIBRATION
# -----------------------------------------------------------------------------
with tabs[3]:
    l, r = st.columns([1.2, 1])
    with l:
        with st.container(border=True):
            card_head("Reliability diagram (held-out test set)", "A perfectly calibrated model sits on the diagonal")
            fig = go.Figure(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="rgba(255,255,255,.35)", dash="dash"), name="perfect", hoverinfo="skip"))
            cmap = {"Bayesian Model Averaging": VIOLET, "Logistic Regression": CYAN, "Random Forest": AMBER, "Naive Bayes": PINK}
            for nm, cl in cmap.items():
                pt = R["probs"][nm][1]
                px, py, _ = B.reliability_curve(R["yte"], pt, 6)
                fig.add_trace(go.Scatter(x=px, y=py, mode="lines+markers", name=nm, line=dict(color=cl, width=3 if nm.startswith("Bayes") else 2), marker=dict(size=8)))
            style_fig(fig, 430)
            fig.update_xaxes(title="mean predicted probability", range=[0, 1])
            fig.update_yaxes(title="observed defect frequency", range=[0, 1])
            plot(fig, "rel")
    with r:
        with st.container(border=True):
            card_head("Proper scoring rules", "Lower is better")
            ct = tab[["Algorithm", "Test Brier", "Test Log-loss", "Test ECE"]].copy()
            gtable(ct, fmt={c: "{:.3f}" for c in ct.columns[1:]}, best={c: "min" for c in ct.columns[1:]})
            st.write("")
            card_head("Distribution of BMA probabilities", "")
            fig = go.Figure()
            fig.add_trace(go.Histogram(x=R["probs"]["Bayesian Model Averaging"][1][R["yte"] == 0], name="clean", marker_color=CYAN, opacity=.7, nbinsx=25))
            fig.add_trace(go.Histogram(x=R["probs"]["Bayesian Model Averaging"][1][R["yte"] == 1], name="defective", marker_color=PINK, opacity=.7, nbinsx=25))
            fig.update_layout(barmode="overlay")
            style_fig(fig, 190)
            plot(fig, "hist")

# -----------------------------------------------------------------------------
# TAB 5 - COMPARISON
# -----------------------------------------------------------------------------
with tabs[4]:
    with st.container(border=True):
        card_head("Model-comparison table", "Same held-out test set · hyper-parameters and decision threshold tuned on validation data only · best value highlighted")
        ct = tab.drop(columns=["Test PR-AUC", "Test Precision", "Test Recall"]).copy()
        ct["Inference time (ms/1000)"] = ct["Inference time (ms/1000)"]
        gtable(ct, fmt={"Validation ROC-AUC": "{:.3f}", "Test ROC-AUC": "{:.3f}", "Test F1": "{:.3f}", "Test Brier": "{:.3f}", "Test Log-loss": "{:.3f}",
                        "Test ECE": "{:.3f}", "Training time (s)": "{:.2f}", "Inference time (ms/1000)": "{:.1f}"},
               best={"Test ROC-AUC": "max", "Validation ROC-AUC": "max", "Test F1": "max", "Test Brier": "min", "Test Log-loss": "min", "Test ECE": "min"},
               wrap=("Interpretability",))
        st.download_button("⬇ Download comparison table (CSV)", tab.to_csv(index=False).encode(), "model_comparison.csv", "text/csv")
    l, r = st.columns(2)
    colmap = {"Bayesian Model Averaging": VIOLET, "Best single BIC model": LILAC, "Logistic Regression": CYAN, "Naive Bayes": PINK, "Random Forest": AMBER}
    with l:
        with st.container(border=True):
            card_head("ROC curves", "")
            fig = go.Figure(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="rgba(255,255,255,.3)", dash="dash"), showlegend=False, hoverinfo="skip"))
            for nm, cl in colmap.items():
                fpr, tpr, _ = roc_curve(R["yte"], R["probs"][nm][1])
                fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{nm} ({roc_auc_score(R['yte'], R['probs'][nm][1]):.3f})", line=dict(color=cl, width=3 if nm.startswith("Bayes") else 1.8)))
            style_fig(fig, 420)
            fig.update_layout(legend=dict(orientation="h", y=-0.28, x=0, font=dict(size=10)))
            fig.update_xaxes(title="false-positive rate")
            fig.update_yaxes(title="true-positive rate")
            plot(fig, "roc")
    with r:
        with st.container(border=True):
            card_head("Precision–recall curves", "Better for imbalanced defect data")
            fig = go.Figure()
            for nm, cl in colmap.items():
                pr, rc, _ = precision_recall_curve(R["yte"], R["probs"][nm][1])
                fig.add_trace(go.Scatter(x=rc, y=pr, mode="lines", name=f"{nm} ({average_precision_score(R['yte'], R['probs'][nm][1]):.3f})", line=dict(color=cl, width=3 if nm.startswith("Bayes") else 1.8)))
            style_fig(fig, 420)
            fig.update_layout(legend=dict(orientation="h", y=-0.28, x=0, font=dict(size=10)))
            fig.update_xaxes(title="recall")
            fig.update_yaxes(title="precision")
            plot(fig, "pr")
    with st.container(border=True):
        card_head("Robustness: repeat the whole study on several random splits",
                  "A single split is noisy on small NASA datasets. This re-runs preprocessing, tuning and testing per seed.")
        nrep = st.slider("Number of random splits", 3, 10, 5)
        if st.button("▶ Run robustness study"):
            with st.spinner(f"Running {nrep} independent splits …"):
                st.session_state["rep"] = (data_key, nrep, get_repeated(data_key, buf.getvalue(), nrep, float(tau)))
        if "rep" in st.session_state and st.session_state["rep"][0] == data_key:
            rep = st.session_state["rep"][2]
            gtable(B.summarise_repeated(rep))
            fig = go.Figure()
            for nm, cl in colmap.items():
                d = rep[rep.Algorithm == nm]
                fig.add_trace(go.Box(y=d["Test ROC-AUC"], name=nm.replace("Bayesian Model Averaging", "BMA").replace("Logistic Regression", "LogReg").replace("Random Forest", "RF").replace("Best single BIC model", "Best-BIC"),
                                     marker_color=cl, line=dict(color=cl), boxmean=True, fillcolor="rgba(139,92,246,.12)"))
            style_fig(fig, 320, legend=False)
            fig.update_yaxes(title="Test ROC-AUC")
            plot(fig, "box")

# -----------------------------------------------------------------------------
# TAB 6 - PLAYGROUND
# -----------------------------------------------------------------------------
with tabs[5]:
    l, r = st.columns([1, 1.2])
    pipk = bma.inclusion_probabilities().head(6).index.tolist()
    Xtr_raw = sp["Xtr"]
    base = Xtr_raw.median()
    vals = {}
    with l:
        with st.container(border=True):
            card_head("Describe a module", "Sliders show the six most influential metrics · all others are set to the training median")
            for c in pipk:
                lo, hi = float(Xtr_raw[c].quantile(0.01)), float(Xtr_raw[c].quantile(0.995))
                if hi <= lo:
                    hi = lo + 1
                isint = bool(np.allclose(Xtr_raw[c].dropna() % 1, 0))
                if isint:
                    vals[c] = st.slider(c, int(lo), int(np.ceil(hi)), int(np.clip(base[c], lo, hi)))
                else:
                    vals[c] = st.slider(c, lo, hi, float(np.clip(base[c], lo, hi)), step=(hi - lo) / 200)
    x = base.copy()
    for k_, v_ in vals.items():
        x[k_] = v_
    Zx = pre.transform(pd.DataFrame([x]))
    dd = bma.predict_dist(Zx, level=level, n_draws=3000, seed=11)
    p, lo_, hi_ = dd["mean"][0], dd["lo"][0], dd["hi"][0]
    tier = "High" if lo_ >= thr else ("Low" if hi_ < thr else "Review")
    with r:
        with st.container(border=True):
            card_head("Live prediction", "Updates instantly as you move the sliders")
            fig = go.Figure(go.Indicator(mode="gauge+number", value=p * 100, number=dict(suffix="%", font=dict(color="#fff", size=44)),
                gauge=dict(axis=dict(range=[0, 100], tickcolor="#8f86bd"), bar=dict(color=TIER_COLOR[tier], thickness=.28), bgcolor="rgba(255,255,255,.05)",
                           borderwidth=0, steps=[dict(range=[lo_ * 100, hi_ * 100], color="rgba(167,139,250,.35)")],
                           threshold=dict(line=dict(color=AMBER, width=3), value=thr * 100))))
            style_fig(fig, 280, legend=False)
            fig.update_layout(margin=dict(l=40, r=40, t=40, b=10))
            plot(fig, "gauge")
            H(f"""<div style="text-align:center;margin-top:-6px">{pill(tier)} &nbsp; <span style="color:#b7aedf">{int(level*100)}% credible interval:
            <b style="color:#fff">{lo_*100:.0f}% – {hi_*100:.0f}%</b> &nbsp;·&nbsp; decision threshold {thr*100:.0f}%</span></div>""")
            fig = go.Figure(go.Histogram(x=dd["draws"][0], nbinsx=45, marker=dict(color=VIOLET, line=dict(width=0)), opacity=.9))
            fig.add_vline(x=thr, line_dash="dot", line_color=AMBER)
            style_fig(fig, 200, legend=False)
            fig.update_xaxes(range=[0, 1], title="posterior predictive of P(defect)")
            plot(fig, "playhist")
            st.caption(f"Epistemic uncertainty {dd['epistemic'][0]:.3f} nats · aleatoric {dd['aleatoric'][0]:.3f} nats. "
                       "A wide interval means the candidate models disagree here — treat the module as 'needs review'.")

# -----------------------------------------------------------------------------
# TAB 7 - ABOUT
# -----------------------------------------------------------------------------
with tabs[6]:
    l, r = st.columns(2)
    with l:
        with st.container(border=True):
            card_head("Method in one minute", "")
            st.markdown(f"""
**Goal.** Identify defect-prone software modules *and* quantify model uncertainty.

**1 · Preprocess (train only).** signed-log1p → drop constant → drop |ρ|>0.95 → z-score. Currently **{len(pre.columns_)}** metrics kept
({len(pre.dropped_correlated_)} correlated dropped).

**2 · Candidate models.** Bayesian logistic regressions over many metric subsets (greedy BIC forward selection + **MC3** search), Gaussian Naive Bayes. {bma.n_visited_} subsets visited, top {len(bma.models_)} kept.

**3 · Posterior model weights.** P(Mₖ|D) ∝ exp(−τ·BICₖ/2). Each logistic member also has a Laplace posterior over its coefficients.

**4 · Average predictions.** p(y=1|x) = Σₖ P(Mₖ|D)·E[p(y=1|x,θₖ,Mₖ)]; credible intervals come from sampling model → coefficients → probability.

**5 · Assess calibration.** Brier, log-loss, ECE, reliability diagrams.
""")
    with r:
        with st.container(border=True):
            card_head("Experimental protocol", "")
            st.markdown(f"""
- Exact duplicate modules removed first (they would leak across splits): **{n_all:,}** unique modules.
- Split **{int((1-test_size-0.2)*100)}/20/{int(test_size*100)}** train / validation / test, stratified, seed **{seed}**.
- **Same held-out test set** for every algorithm; preprocessing fitted on training only.
- Hyper-parameters (LogReg C, RF depth/leaf, NB smoothing) and the F1 decision threshold are tuned on the validation set only. BMA itself needs no validation data.
- Comparison algorithms: Logistic Regression, Random Forest, Naive Bayes (+ best single BIC model as an ablation).
- Metrics: ROC-AUC, F1, Brier score, log-loss (+ PR-AUC, ECE).
- NASA MDP data has no timestamps or entity IDs, so a stratified random split is used; this can be optimistic versus a real time-ordered deployment.
""")
    H('<div class="footer">DefectLens · Bayesian Model Averaging · NASA / PROMISE defect datasets · built with Streamlit &amp; Plotly</div>')

In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)
open(".streamlit/config.toml", "w").write('''[theme]
base = "dark"
primaryColor = "#8b5cf6"
backgroundColor = "#05030f"
secondaryBackgroundColor = "#140b36"
textColor = "#e9e5ff"
font = "sans serif"

[server]
headless = true
enableCORS = false
enableXsrfProtection = false

[browser]
gatherUsageStats = false
''')
print("app.py + theme written")

In [ ]:
LAUNCH_DEMO = True
import subprocess, re, time, os, sys
if LAUNCH_DEMO:
    subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)
    if not os.path.exists("cloudflared"):
        os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared")
    st_proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                                "--server.enableCORS", "false", "--server.enableXsrfProtection", "false", "--browser.gatherUsageStats", "false"],
                               stdout=open("st.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(6)
    tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
    url = None
    for line in tun.stderr:
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if m: url = m.group(0); break
    print("🔮 DefectLens is live ➜", url)
    print("(open the link; first load takes ~10 s while the models are fitted. Stop this cell to shut it down.)")
    try:
        from IPython.display import display, HTML
        display(HTML(f'<a href="{url}" target="_blank" style="display:inline-block;padding:12px 22px;border-radius:12px;background:#7c3aed;color:white;font-weight:600;text-decoration:none">Open DefectLens ↗</a>'))
    except Exception: pass
    tun.wait()